# Lección 4.1 — Métricas de regresión: ¿por cuánto me equivoco?

> **Módulo 4 · Evaluación: ¿mi modelo es bueno?** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 80–100 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.5 (división entrenamiento/prueba), Lección 3.2 (función de pérdida)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es un **error** (residuo) y por qué hay que resumir muchos errores en un solo número.
- Calcular e interpretar **MAE**, **RMSE** y **R²** con tus propias palabras.
- Predecir cuándo el RMSE se dispara mientras el MAE casi no se mueve (los **atípicos**).
- Elegir la métrica de regresión adecuada y evitar los errores más comunes al reportarla.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error,
                             r2_score, mean_absolute_percentage_error)

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el pronóstico del tiempo de entrega

Pides comida por una app y te dice: **"llega en 30 minutos"**. Llega en 37. La app se equivocó por **7 minutos**.

Un solo pedido no dice mucho: a veces tienes suerte y a veces no. Para saber si la app es **buena prediciendo**, tendrías que mirar **cientos de pedidos** y resumir todos esos errores en **un solo número** que puedas comparar:

```
Pedido 1:  predijo 30, llegó en 37  → error  +7
Pedido 2:  predijo 25, llegó en 22  → error  -3
Pedido 3:  predijo 40, llegó en 41  → error  +1
   ...            ...                     ...
                                  ¿Resumen? → "en promedio me equivoco por ___ minutos"
```

Eso es una **métrica de regresión**: una forma de resumir los errores de un modelo que predice **números** (minutos, precios, temperaturas…). Hay varias formas de resumirlos, y **cada una cuenta una historia distinta**.

### ✋ Ejercicio 1 — Calentamiento
La app A se equivocó en tres pedidos por **+5, −5 y +5** minutos. La app B, por **+1, +1 y +1**.

Si para resumir **sumas los errores tal cual** (con su signo), ¿qué app parece mejor? ¿Es justo?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Sumando con signo: A da **+5** (5 − 5 + 5) y B da **+3**. Parece que la diferencia es pequeña, pero A se equivoca **5 veces más** en cada pedido. Peor aún: una app que se equivoca por +30 y −30 sumaría **0** y parecería perfecta.

Los errores positivos y negativos **se cancelan**. Por eso todas las métricas que verás **quitan el signo**: con el valor absoluto (MAE) o elevando al cuadrado (RMSE).
</details>

---
## 2. 📖 El concepto: error (residuo) y MAE

Para cada ejemplo:

> **error** (o **residuo**) = valor real − valor predicho

La métrica más sencilla es el **MAE** (*Mean Absolute Error*, error absoluto medio):

> **MAE** = promedio de los errores **sin signo**

Se lee de forma directa: *"en promedio, me equivoco por MAE minutos"*. Está en **las mismas unidades** que lo que predices.

Vamos a crear los datos de una app de reparto: 600 pedidos con la distancia, si era hora pico y si llovía.

In [ ]:
# ▶️ Historial ficticio de pedidos
rng = np.random.default_rng(4)
n = 600
pedidos = pd.DataFrame({
    "distancia_km": rng.uniform(0.5, 10, n).round(1),
    "hora_pico":    rng.integers(0, 2, n),               # 1 = sí
    "lluvia":       (rng.random(n) < 0.25).astype(int),  # 1 = llovía
})
# El tiempo "real" (el modelo NO conoce esta fórmula) + algo de azar
pedidos["minutos"] = (10 + 3 * pedidos.distancia_km + 8 * pedidos.hora_pico
                      + 6 * pedidos.lluvia + rng.normal(0, 4, n)).round(1)

X = pedidos.drop(columns="minutos")
y = pedidos["minutos"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
pedidos.head()

Entrenamos una regresión lineal (Lección 5.1) y miramos sus errores en **5 pedidos de prueba** (pedidos que el modelo no vio al entrenar).

In [ ]:
modelo = LinearRegression().fit(X_tr, y_tr)
pred = modelo.predict(X_te)

vistazo = pd.DataFrame({"real": y_te.values, "predicho": pred.round(1)})
vistazo["error"] = (vistazo.real - vistazo.predicho).round(1)
vistazo["error_sin_signo"] = vistazo.error.abs()
vistazo.head()

### ✋ Ejercicio 2 — Calcula el MAE a mano
Usa **solo los 5 pedidos** de la tabla de arriba. Suma la columna `error_sin_signo` y divide entre 5.

✍️ *Mi MAE de 5 pedidos:*

Luego ejecuta la celda para comprobarlo, y mira el MAE con **todos** los pedidos de prueba.

In [ ]:
print(f"MAE con los 5 primeros pedidos: {vistazo.error_sin_signo.head().mean():.2f} min")
print(f"MAE con los {len(y_te)} pedidos de prueba: {mean_absolute_error(y_te, pred):.2f} min")

<details><summary><b>👀 Ver respuesta</b></summary>

Errores sin signo: 3.1 + 0.7 + 1.5 + 4.0 + 6.8 = 16.1 → 16.1 / 5 ≈ **3.22 min**.

Con los 150 pedidos de prueba el MAE es **≈ 3.24 min**: *"en promedio, el modelo se equivoca por unos 3 minutos"*. Fíjate en que 5 pedidos dieron casi lo mismo por casualidad: con tan pocos ejemplos la cifra puede bailar mucho. **Mide siempre con muchos ejemplos.**
</details>

---
## 3. 📖 RMSE: castigar los errores grandes

La segunda métrica estrella es el **RMSE** (*Root Mean Squared Error*, raíz del error cuadrático medio). Se calcula en tres pasos, que son su nombre al revés:

1. **S**quared: eleva cada error **al cuadrado** (así desaparece el signo).
2. **M**ean: saca el **promedio** de esos cuadrados (eso es el **MSE**, que viste como función de pérdida en la Lección 3.2).
3. **R**oot: saca la **raíz cuadrada** para volver a las unidades originales (minutos).

¿Por qué tanto lío? Por el paso 1: al elevar al cuadrado, **un error de 10 pesa 100, pero uno de 2 pesa solo 4**. El RMSE **castiga mucho más los errores grandes**.

| Error | Sin signo (MAE) | Al cuadrado (RMSE) |
|---|---|---|
| 1 | 1 | 1 |
| 2 | 2 | 4 |
| 10 | 10 | **100** |

### 🔮 Ejercicio 3 — Dos apps con el mismo MAE
En 10 pedidos:
- La **app A** se equivoca por **2 minutos** en todos.
- La **app B** acierta exacto en 9 y se equivoca por **20 minutos** en uno.

1. ¿Cuál es el MAE de cada una?
2. ¿Cuál tendrá **mayor RMSE**?
3. Como cliente, ¿cuál prefieres?

✍️ *Mis predicciones:*

In [ ]:
real = np.full(10, 30.0)                      # 10 pedidos que tardaron 30 min
app_A = real + 2                               # siempre 2 min de error
app_B = real.copy(); app_B[-1] = real[-1] + 20  # 9 perfectos y uno con 20 min de error

for nombre, p in [("App A", app_A), ("App B", app_B)]:
    print(f"{nombre}: MAE = {mean_absolute_error(real, p):.2f}   RMSE = {root_mean_squared_error(real, p):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Las dos tienen **MAE = 2.00**: el promedio de errores sin signo es igual.
2. La B tiene **RMSE ≈ 6.32**, contra 2.00 de la A. El error de 20 minutos, al cuadrado, pesa 400 y domina el promedio.
3. Depende del negocio, y esa es la clave de la Lección 4.8. Si un retraso de 20 minutos significa comida fría y un cliente perdido, prefieres la A, y el **RMSE** es la métrica que lo refleja. Si solo te importa el error típico, el **MAE** dice que empatan.

> 📝 Regla rápida: si **RMSE ≈ MAE**, los errores son parejos. Si **RMSE ≫ MAE**, hay **unos pocos errores muy grandes** escondidos.
</details>

### 🔮 Ejercicio 4 — Llegan los atípicos
En la vida real hay pedidos raros: un accidente, un repartidor perdido… Vamos a simular que **6 de los 150 pedidos de prueba** (el 4 %) tardaron **40 minutos más** de lo normal. El modelo es **el mismo**: solo cambian los datos reales.

Antes de ejecutar: ¿qué métrica crees que se moverá más, el MAE o el RMSE? ¿Más o menos cuánto?

✍️ *Mi apuesta:*

In [ ]:
y_te_raro = y_te.copy()
rng_raro = np.random.default_rng(1)
y_te_raro.iloc[rng_raro.choice(len(y_te), 6, replace=False)] += 40   # ✏️ prueba +80 o más pedidos

for nombre, real in [("Sin atípicos", y_te), ("Con 6 atípicos", y_te_raro)]:
    print(f"{nombre:<15} MAE = {mean_absolute_error(real, pred):5.2f}   RMSE = {root_mean_squared_error(real, pred):5.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- MAE: de **3.24 a ≈ 4.59** (sube ≈ 1.4 min).
- RMSE: de **3.96 a ≈ 8.46** (¡más del doble!).

Solo el 4 % de los pedidos cambió, pero el RMSE se disparó. Esto es útil para **detectar** que hay casos extremos, y peligroso si esos casos son errores de captura (un "400 minutos" que en realidad eran 40): el RMSE te asustará por culpa de unos pocos datos sucios (Lección 2.2).
</details>

---
## 4. 📖 R²: ¿cuánto mejor que adivinar el promedio?

MAE y RMSE están en minutos. Pero ¿un MAE de 3 minutos es bueno? Depende: si los pedidos tardan entre 30 y 35 minutos, es malísimo; si tardan entre 10 y 60, es muy bueno.

El **R²** (*coeficiente de determinación*) responde comparando tu modelo con el modelo **más tonto posible**: el que siempre predice **el promedio**.

| Valor de R² | Significa |
|---|---|
| **1** | Predicción perfecta |
| **0** | Igual que predecir siempre el promedio: el modelo no aporta nada |
| **entre 0 y 1** | Mejor que el promedio (p. ej. 0.85 = elimina el 85 % del error cuadrático del "promedio") |
| **negativo** | **¡Peor que predecir siempre el promedio!** |

> Por dentro: R² = 1 − (error cuadrático del modelo) / (error cuadrático de predecir el promedio).

No tiene unidades, así que sirve para **comparar** sin pensar en minutos.

### 🔮 Ejercicio 5 — Tres modelos, tres R²
Vamos a medir el R² en el conjunto de prueba de:
1. Nuestro modelo de regresión lineal.
2. Un "modelo" que siempre predice el **promedio del entrenamiento**.
3. Un "modelo" que siempre predice **60 minutos**.

Ordena tus apuestas: ¿cuál tendrá un R² cerca de 0? ¿Alguno será negativo?

✍️ *Mis predicciones:*

In [ ]:
promedio = np.full(len(y_te), y_tr.mean())
siempre_60 = np.full(len(y_te), 60.0)   # ✏️ prueba con 33 (casi el promedio)

for nombre, p in [("Regresión lineal", pred), ("Siempre el promedio", promedio), ("Siempre 60 min", siempre_60)]:
    print(f"{nombre:<20} R² = {r2_score(y_te, p):6.2f}   MAE = {mean_absolute_error(y_te, p):5.2f} min")

<details><summary><b>👀 Ver respuesta</b></summary>

- Regresión lineal: **R² ≈ 0.85**. Elimina la mayor parte del error que tendría "adivinar el promedio".
- Siempre el promedio: **R² ≈ −0.02**, prácticamente 0. No es exactamente 0 porque usamos el promedio del **entrenamiento** y lo medimos en **prueba**.
- Siempre 60 min: **R² ≈ −7.13**. ¡Muy negativo! Es mucho peor que predecir el promedio.

**Sí, el R² puede ser negativo** en datos de prueba. Si te pasa con un modelo de verdad, es una alarma: tu modelo pierde contra la estrategia más tonta. Comparar contra estas estrategias tontas tiene nombre propio: **líneas base** (Lección 4.7).
</details>

### 🔮 Ejercicio 6 — Cambiar de unidades
El jefe pide el reporte **en horas** en lugar de minutos. Dividimos entre 60 los valores reales y las predicciones.

¿Qué les pasa al MAE, al RMSE y al R²?

✍️ *Mi apuesta:*

In [ ]:
for unidad, factor in [("minutos", 1), ("horas", 60)]:
    r, p = y_te / factor, pred / factor
    print(f"En {unidad:<8} MAE = {mean_absolute_error(r, p):.3f}   "
          f"RMSE = {root_mean_squared_error(r, p):.3f}   R² = {r2_score(r, p):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

MAE y RMSE **se dividen entre 60** (3.239 → 0.054 horas; 3.962 → 0.066 horas): están en las unidades de lo que predices. El **R² no cambia** (0.848): es una proporción sin unidades.

Por eso, al reportar MAE o RMSE **siempre di la unidad** ("3.2 minutos", no "3.2"). Y nunca compares el MAE de un modelo que predice en pesos con otro que predice en dólares.
</details>

---
## 5. 📖 Un primo útil y tramposo: el MAPE

A los gerentes les encantan los porcentajes. El **MAPE** (*Mean Absolute Percentage Error*) mide el error **en porcentaje del valor real**:

> "en promedio me equivoco por un 11 % del tiempo real"

Es fácil de comunicar… pero tiene una trampa: cuando el valor real es **pequeño**, cualquier error se vuelve un porcentaje enorme.

### 🔮 Ejercicio 7 — La trampa del MAPE
Ahora el modelo predice **minutos de retraso**. Dos pedidos:
- Pedido 1: retraso real **1 min**, predicho **3 min** (error de 2 min).
- Pedido 2: retraso real **50 min**, predicho **52 min** (error de 2 min).

Los dos errores son de 2 minutos. ¿Cuál es el MAE? ¿Y el MAPE?

✍️ *Mi apuesta:*

In [ ]:
real_retraso = [1, 50]
pred_retraso = [3, 52]
print(f"MAE  = {mean_absolute_error(real_retraso, pred_retraso):.1f} min")
print(f"MAPE = {mean_absolute_percentage_error(real_retraso, pred_retraso):.0%}")
print(f"MAPE de nuestro modelo de entregas: {mean_absolute_percentage_error(y_te, pred):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **MAE = 2.0 min**: los dos pedidos tienen el mismo error.
- **MAPE = 102 %**: el pedido 1 aporta un error del 200 % (2 de 1) y el pedido 2 solo del 4 % (2 de 50). El promedio es 102 %, y un gerente concluiría que el modelo es pésimo.

Con el tiempo total de entrega (valores lejos de 0) el MAPE sí es razonable: **≈ 11 %**. Regla: **usa MAPE solo si los valores reales nunca están cerca de 0** (si alguno es 0, ni siquiera se puede calcular bien).
</details>

### 📖 ¿Cuál uso cuándo?

| Métrica | Se lee como… | Unidades | Atípicos | Úsala cuando… |
|---|---|---|---|---|
| **MAE** | "en promedio me equivoco por X" | Las del objetivo | Robusta: pesan lo normal | Todos los errores cuestan parecido; quieres algo fácil de explicar |
| **RMSE** | "error típico, castigando los grandes" | Las del objetivo | Sensible: pesan mucho | Un error grande es mucho peor que varios pequeños |
| **R²** | "qué parte del error del promedio elimino" | Sin unidades (≤ 1) | Sensible | Quieres saber si el modelo aporta algo frente a adivinar el promedio |
| **MAPE** | "en promedio me equivoco un X %" | Porcentaje | Explota con valores reales pequeños | Público no técnico y valores siempre lejos de 0 |

> 💡 No se elige una sola: lo normal es reportar **MAE (o RMSE) + R²**. Uno dice "cuánto en unidades reales"; el otro, "cuánto mejor que no tener modelo". Elegir **la principal** depende del costo de los errores (Lección 4.8).

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué hacer |
|---|---|---|
| Métricas espectaculares (R² ≈ 1, MAE ≈ 0) | Mediste en **entrenamiento**, o hay fuga de datos | Mide siempre en datos que el modelo no vio (Lecciones 2.5, 2.6) |
| R² negativo en prueba | El modelo es peor que predecir el promedio | Revisa sobreajuste (Lección 4.4) y compara con líneas base (Lección 4.7) |
| RMSE mucho mayor que el MAE | Unos pocos errores enormes | Mira esos casos: ¿son atípicos reales o datos sucios? (Lección 2.2) |
| "Mi R² de 0.4 es malo, el de mi colega es 0.9" | Se comparan R² de **problemas distintos** | El R² solo se compara entre modelos **del mismo conjunto de datos** |
| "RMSE = 3.96" sin más | Falta la unidad | Di siempre la unidad: "3.96 minutos" |
| MAPE gigantesco | Valores reales cerca de 0 | Usa MAE o RMSE |

El primero es el más frecuente. Veámoslo.

### 🔮 Ejercicio 8 — El examen con las respuestas
Entrenamos un árbol de regresión **sin límites** (Lección 5.4) con los pedidos y lo medimos dos veces: con los pedidos de **entrenamiento** y con los de **prueba**.

¿Cuál de los dos números le reportarías a tu jefe? ¿Qué tan distintos crees que serán?

✍️ *Mi apuesta:*

In [ ]:
arbol = DecisionTreeRegressor(random_state=0).fit(X_tr, y_tr)

for nombre, Xs, ys in [("Entrenamiento", X_tr, y_tr), ("Prueba", X_te, y_te)]:
    p = arbol.predict(Xs)
    print(f"{nombre:<14} MAE = {mean_absolute_error(ys, p):.2f} min   R² = {r2_score(ys, p):.2f}")
print(f"\n(Regresión lineal en prueba: MAE = {mean_absolute_error(y_te, pred):.2f} min)")

<details><summary><b>👀 Ver respuesta</b></summary>

En entrenamiento el árbol parece excelente (**MAE ≈ 1.71 min, R² ≈ 0.94**), pero en prueba empeora mucho (**MAE ≈ 3.93 min, R² ≈ 0.76**), e incluso pierde contra la regresión lineal (3.24 min). El árbol memorizó los pedidos que ya vio.

**Al jefe se le reporta siempre la cifra de prueba.** Medir en entrenamiento es como calificar un examen que el alumno ya tenía resuelto. (Ese hueco entre entrenamiento y prueba es el **sobreajuste**: Lección 4.4.)
</details>

---
## 7. 🧠 Resumen en 5 puntos
1. Un **error** (residuo) es real − predicho; para juzgar un modelo se resumen **muchos** errores en una métrica, siempre **en datos que no vio**.
2. **MAE** = error promedio sin signo: "en promedio me equivoco por X unidades". Fácil de explicar y robusto a atípicos.
3. **RMSE** eleva los errores al cuadrado antes de promediar: **castiga los errores grandes**. Si RMSE ≫ MAE, hay errores enormes escondidos.
4. **R²** compara con predecir siempre el promedio: 1 = perfecto, 0 = no aporta nada, **negativo = peor que el promedio**. No tiene unidades.
5. Reporta **la unidad**, no compares R² entre problemas distintos y desconfía del **MAPE** cuando hay valores reales cerca de 0.

---
## 8. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 8.1 Repaso rápido (opción múltiple)

**1.** Un modelo que predice el precio de casas tiene MAE = $40,000 y RMSE = $160,000. ¿Qué es lo más probable?
- a) Se equivoca por $160,000 en casi todas las casas
- b) La mayoría de los errores son moderados, pero hay unas pocas casas con errores enormes
- c) El modelo es perfecto
- d) Hay un error de cálculo: el RMSE nunca puede ser mayor que el MAE

**2.** Tu modelo tiene R² = −0.3 en el conjunto de prueba. Significa que…
- a) Explica el 30 % de la variación
- b) Es peor que predecir siempre el promedio
- c) Hay un error: el R² no puede ser negativo
- d) Está sobreajustado con certeza

**3.** Pasas tus predicciones de metros a centímetros. ¿Qué métrica **no** cambia?
- a) MAE · b) RMSE · c) R² · d) MSE

**4.** Predices el número de unidades vendidas por producto y muchos productos venden 0 o 1 unidad al día. ¿Qué métrica **evitarías** como principal?
- a) MAE · b) RMSE · c) MAPE · d) R²

**5.** En una aseguradora, subestimar por mucho el costo de un siniestro grande es desastroso; los errores pequeños dan igual. ¿Qué métrica refleja mejor esa preocupación?
- a) MAE · b) RMSE · c) MAPE · d) Exactitud

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** RMSE ≫ MAE es la firma de unos pocos errores muy grandes. Sobre d): el RMSE **siempre** es mayor o igual que el MAE; son iguales solo si todos los errores miden lo mismo.
2. **b)** Un R² negativo significa que el modelo comete más error cuadrático que la línea base "siempre el promedio". Puede deberse a sobreajuste, pero no es la única causa (por eso d) no es segura).
3. **c)** El R² es una proporción sin unidades. MAE y RMSE se multiplican por 100; el MSE, por 10,000.
4. **c)** Con valores reales de 0 o 1, el MAPE explota (o ni se puede calcular).
5. **b)** El RMSE eleva al cuadrado los errores, así que los grandes pesan muchísimo más. (La exactitud es de clasificación: Lección 4.2.)
</details>

### 8.2 Preguntas para pensar (sin ejecutar)

**A.** Un compañero dice: *"Mi modelo de precios de autos tiene R² = 0.92 y el tuyo de demanda eléctrica solo 0.75, así que el mío es mejor"*. ¿Qué le contestas?

**B. Caso de análisis 💼** Una cadena de panaderías predice cuántos panes hornear cada día por sucursal. El modelo A tiene MAE = 12 panes y RMSE = 15. El modelo B tiene MAE = 10 panes y RMSE = 40. El gerente dice que **quedarse sin pan** un día de mucha demanda es gravísimo (clientes que no vuelven), pero que sobrar 10 panes casi no importa. ¿Qué modelo recomiendas y por qué? ¿Qué te gustaría saber además?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Que el R² **no se compara entre problemas distintos**. Cada R² mide la mejora frente al promedio *de su propio conjunto de datos*: hay problemas donde el promedio ya es muy malo (fácil superarlo) y otros donde hay mucho azar que ningún modelo puede predecir. Solo tiene sentido comparar R² de modelos distintos **sobre los mismos datos de prueba**.

**B.** Un buen argumento: el **A**. Aunque B tiene mejor MAE, su RMSE (40) mucho mayor que su MAE (10) indica que **a veces falla por muchísimo**, justo lo que el gerente teme. A es más parejo (RMSE ≈ MAE). Información útil: ¿los errores grandes de B son por **quedarse corto** o por **pasarse**? MAE y RMSE no distinguen la **dirección** del error, y aquí importa mucho; habría que mirar los errores por separado. (Elegir métrica por el costo del error es el tema de la Lección 4.8.)
</details>

### 8.3 🧩 Práctica integradora: "¿Qué tan rápido avanzará la diabetes?"

Un hospital tiene datos de **442 pacientes con diabetes** (edad, índice de masa corporal, presión arterial y seis análisis de sangre) y una medida de **cuánto avanzó la enfermedad un año después** (un número entre 25 y 346; más alto = peor). Quieren saber si vale la pena un modelo para anticipar qué pacientes empeorarán más.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
X_d, y_d = load_diabetes(return_X_y=True, as_frame=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_d, y_d, test_size=0.25, random_state=0)

print(f"Pacientes: {len(X_d)} ({len(Xd_tr)} de entrenamiento, {len(Xd_te)} de prueba)")
print(f"Avance de la enfermedad: promedio {y_d.mean():.0f}, mínimo {y_d.min():.0f}, máximo {y_d.max():.0f}")
plt.hist(y_d, bins=30, edgecolor="k"); plt.xlabel("Avance de la enfermedad"); plt.ylabel("Pacientes"); plt.show()

**Paso 2 🔮 — La línea base.** Antes de entrenar nada, calcula las métricas de predecir siempre el **promedio del entrenamiento**. ¿Qué R² esperas? ¿Y qué MAE, aproximadamente, mirando el histograma?

✍️ *Mi apuesta:*

In [ ]:
def reporte(nombre, real, predicho):
    print(f"{nombre:<22} MAE = {mean_absolute_error(real, predicho):5.1f}   "
          f"RMSE = {root_mean_squared_error(real, predicho):5.1f}   R² = {r2_score(real, predicho):5.2f}")

reporte("Siempre el promedio", yd_te, np.full(len(yd_te), yd_tr.mean()))

<details><summary><b>👀 Ver respuesta</b></summary>

**R² ≈ 0.00**, como era de esperar, y **MAE ≈ 58.3**: adivinando el promedio nos equivocamos, en promedio, por unos 58 puntos de avance. Esa es la cifra que cualquier modelo **tiene que mejorar** para valer la pena (Lección 4.7).
</details>

**Paso 3 ▶️ — Tres candidatos.** Entrenamos una regresión lineal, un árbol pequeño (`max_depth=3`) y un árbol sin límites. Ejecuta y llena la tabla mentalmente: ¿quién gana en cada métrica?

In [ ]:
candidatos = {
    "Regresión lineal":      LinearRegression(),
    "Árbol max_depth=3":     DecisionTreeRegressor(max_depth=3, random_state=0),
    "Árbol sin límites":     DecisionTreeRegressor(random_state=0),   # ✏️ prueba max_depth=2 o 5
}
for nombre, m in candidatos.items():
    m.fit(Xd_tr, yd_tr)
    reporte(nombre, yd_te, m.predict(Xd_te))

✋ Responde:
1. ¿Qué modelo gana? ¿Gana en las tres métricas a la vez?
2. ¿Hay algún modelo **peor que la línea base**? ¿Cómo lo sabes con una sola cifra?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Gana la **regresión lineal** en las tres: MAE ≈ 45.1, RMSE ≈ 56.4, R² ≈ 0.36. El árbol de profundidad 3 queda en medio (MAE ≈ 52.6, R² ≈ 0.09).
2. Sí: el **árbol sin límites** tiene **R² ≈ −0.18**, peor que adivinar el promedio. Su MAE (≈ 60.4) también es peor que el de la línea base (58.3).
</details>

**Paso 4 ✋ — ¿Un R² de 0.36 es malo?** En la sección de reparto obtuvimos R² ≈ 0.85. ¿Significa que este modelo es "malo"? ¿Qué le dirías al hospital?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

No se pueden comparar: son **problemas distintos**. Predecir cómo evolucionará una enfermedad con 10 mediciones es mucho más difícil (hay muchísimo azar y factores no medidos) que predecir un tiempo de entrega que depende casi solo de la distancia.

Lo que sí podemos decir: el modelo lineal reduce el error típico de **≈ 58 a ≈ 45 puntos** (un 23 % menos que adivinar el promedio). Útil para ordenar a los pacientes por riesgo, **pero no lo bastante preciso** para tomar decisiones individuales sin el criterio de un médico.
</details>

**Paso 5 🔮 — El autoengaño.** Mide el árbol sin límites en **entrenamiento**. ¿Qué R² esperas?

✍️ *Mi apuesta:*

In [ ]:
sin_limites = candidatos["Árbol sin límites"]
reporte("Árbol (entrenamiento)", yd_tr, sin_limites.predict(Xd_tr))
reporte("Árbol (prueba)", yd_te, sin_limites.predict(Xd_te))

<details><summary><b>👀 Ver respuesta</b></summary>

En entrenamiento: **MAE = 0 y R² = 1.00**, ¡perfecto! En prueba: R² ≈ −0.18. El árbol memorizó a los 331 pacientes de entrenamiento. Si alguien te presentara solo la primera línea, creerías tener un modelo perfecto cuando en realidad es peor que no tener modelo.

**Moraleja de toda la lección:** una métrica solo significa algo si sabes **en qué datos** se midió y **contra qué** la comparas.
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "R-squared, Clearly Explained" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Regression metrics* (`mean_absolute_error`, `root_mean_squared_error`, `r2_score`…).
- 📖 **Lectura:** Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, capítulo "Evaluating point forecast accuracy" (gratis en línea): por qué el MAPE da problemas.

---
⬅️ **Anterior:** [3.8 Proyecto del módulo: Entrenador a mano 🩺⚙️](../m03-como-aprende/3.8-proyecto-del-modulo.md) · ➡️ **Siguiente:** [4.2 Métricas de clasificación: más allá de la exactitud](4.2-metricas-de-clasificacion.ipynb)